In [ ]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage,RemoveMessage,BaseMessage
from typing import TypedDict,List,Annotated
from langgraph.graph import StateGraph,add_messages,MessagesState,END,START
from Prompts import Supervisor_prompt_template
import uuid
from langgraph.types import Send,Command
import operator
from langgraph.checkpoint.memory import InMemorySaver
llm = ChatOpenAI(model = 'gpt-4o-mini',api_key=" ")
checkpointer = InMemorySaver()


In [2]:
#Output Schema
class output_state(MessagesState):
    tool_call : bool
    input :str
    output : Annotated[List[BaseMessage],add_messages]
     

class Action_Schema(TypedDict):
    Agent_Name: str = Field(default=None, description='Name of the agent to be invoked')
    message : str = Field(default=None, description='data to be passed to agent')

class output_schema(TypedDict):
    output : str = Field(default = None, description = 'message to respond to user, default value if tool call is required')
    action : List[Action_Schema] = Field(default = None, description = 'list of dictionary with key value pair of Agent_Name and message')

In [3]:

def supervisor_node(state: output_state):

    print(f"\n\nSUPERVISOR NODE: {state}\n\n")

    last_message = state['messages'][-1]


    # Normal flow
    response_model = llm.with_structured_output(output_schema).invoke([
        HumanMessage(
            content=Supervisor_prompt_template.format(
                chat_history=state['messages'][:-1],
                message=last_message
            )
        )
    ])
    for i in state['messages']:
        print(f"\n**MESSAGE** : {i}\n")
    print((f"\n**MESSAGE** : {response_model}\n"))
    if response_model['output']:
        # Chat-only → update state

        print('**NORMAL CHAT**')

        state['messages'].append(AIMessage(content=response_model['output']))
        return state
    else:
        # Action-only → route to specific agent

        print('**AGENT CHAT**')

        state['messages'] = AIMessage(content="Invoking Agent to Complete Tasks")
        sends = [
            Send(resp['Agent_Name'], {'input': resp['message']})
            for resp in response_model['action']
        ]
        return sends

# Example Nodes
def Complaint_Node(state: dict) -> ToolMessage:
    print(f"\n\nCOMPLAINT NODE: {state}\n\n")
    output = f"Complaint has been raised for: {state['input']}"
    print(output)
    return {'output':[ToolMessage(content=output,tool_call_id = uuid.uuid4())]}

def Query_Node(state: dict) -> ToolMessage:
    print(f"\n\nQUERY NODE: {state}\n\n")
    output = f"Query processed for : {state['input']}"
    print(output)
    return {'output':[ToolMessage(content=output,tool_call_id = uuid.uuid4())]}

def Request_Node(state: dict) -> ToolMessage:
    print(f"\n\nREQUEST NODE : {state}\n\n")
    output = f"document has been successfully downloaded"
    print(output)
    return {'output':[ToolMessage(content=output,tool_call_id = uuid.uuid4())]}

def Knowledge_Node(state: dict) -> ToolMessage:
    print(f"\n\nKNOWLEDGE NODE :{state}\n\n")
    output = f"Knowledge fetched for: {state['input']}"
    print(output)
    return {'output':[ToolMessage(content=output,tool_call_id = uuid.uuid4())]}


def Collector_Node(state: dict) -> dict:
    print(f"\n\nCOLLECTOR NODE :{state}\n\n")
    """
    Collect all ToolMessages from state['output'], merge their content,
    and append as a single AIMessage to state['output'].
    Original ToolMessages are removed to avoid duplicates.
    """
    output_list = state.get('output', [])
    tool_messages = [msg for msg in output_list if isinstance(msg, ToolMessage)]

    if not tool_messages:
        return state  # nothing to merge

    remove_message = [RemoveMessage(id = m.id) for m in state['output']]

    # Merge content of all ToolMessages
    merged_content = "\n".join(msg.content for msg in tool_messages)

    return {'messages' : ToolMessage(content=merged_content,tool_call_id = uuid.uuid4()),'output' :remove_message}


In [4]:
supervsior = StateGraph(output_state)

supervsior.add_node('supervisor',supervisor_node)
supervsior.add_node('Complaint_Agent',Complaint_Node)
supervsior.add_node('Query_Agent',Query_Node)
supervsior.add_node('Document_Download_Agent',Request_Node)
supervsior.add_node('Data_Agent',Knowledge_Node)
supervsior.add_node('Collector_Node',Collector_Node)

supervsior.add_conditional_edges(START,supervisor_node,['Complaint_Agent','Query_Agent','Document_Download_Agent','Data_Agent',END])
supervsior.add_edge('Complaint_Agent','Collector_Node')
supervsior.add_edge('Query_Agent','Collector_Node')
supervsior.add_edge('Document_Download_Agent','Collector_Node')
supervsior.add_edge('Data_Agent','Collector_Node')
supervsior.add_edge('Collector_Node','supervisor')
supervsior.add_edge('supervisor',END)

compiled_supervisor = supervsior.compile(checkpointer=checkpointer)

In [5]:
config = {'configurable': {'thread_id' : '1'}}
# user_input = 'download civil report for application id QAWS23EDFR45 and i am frustrated as i havent recieved my document yet'
user_input = 'hyy'
input_data = {'messages' : [HumanMessage(content = user_input)],'tool_call': False}

supervisor_response = compiled_supervisor.invoke(input_data,config = config)



SUPERVISOR NODE: {'messages': [HumanMessage(content='hyy', additional_kwargs={}, response_metadata={}, id='5f668aed-a19b-4df2-baff-cf3faca67ed5')], 'tool_call': False, 'output': []}



**MESSAGE** : content='hyy' additional_kwargs={} response_metadata={} id='5f668aed-a19b-4df2-baff-cf3faca67ed5'


**MESSAGE** : {'output': 'Hello! How can I assist you today?', 'action': None}

**NORMAL CHAT**


TypeError: unhashable type: 'dict'

In [ ]:
supervisor_response

In [ ]:
# content = ' download my document vm summary for application id 124 and download vm summary for application id 2345'
# chat_history=''
# supervisor_output = llm.with_structured_output(output_schema).invoke([HumanMessage(content = Supervisor_prompt_template.format(chat_history=chat_history,message = content))])
# print(supervisor_output)
# if supervisor_output['output'] is not None:
#     print(f'BOT RESPONSE : {supervisor_output['output']}')
# elif supervisor_output['action'] is not None:
#     print(f'AGENT INVOKED : {supervisor_output['action']}')